In [ ]:
#Libs
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException, StaleElementReferenceException,
    UnexpectedAlertPresentException, NoAlertPresentException, ElementClickInterceptedException
)
from datetime import datetime
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote, urljoin
from pathlib import Path
from tqdm import tqdm
import os, re, mimetypes, time, unicodedata, win32com.client, requests, urllib3, pandas as pd, openpyxl
import tempfile, subprocess, shutil, winsound, traceback, json

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [ ]:
# ===== CONFIGURAÇÃO ESPECIAL (AcessosEspeciais) =====
# Curso(s) fixo(s) a serem liberados para TODOS os alunos da planilha.
# Pode-se passar uma lista (ex.: ["312", "565"]).
CURSOS_FIXOS = ["312"]

# Idioma/Site fixo. "PT" -> value="1" | "ESP" -> value="2"
IDIOMA_FIXO = "PT"

ARQUIVO_ACESSOS_ESPECIAIS = "AcessosEspeciais.xlsx"

# ==== CONFIGURAÇÃO ====
URL_LOGIN = "https://gehealthcare.sistematutor.com.br/tutor/sistema/login.do?method=logout"
EMAIL = "gabriel.santos@gehealthcare.com"
SENHA = "Golias123@"

# ==== CHROME ====
chrome_options = Options()
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
chrome_options.add_experimental_option('prefs', {
    'profile.password_manager_enabled': False,
    'credentials_enable_service': False,
    'profile.default_content_setting_values.notifications': 2,
    'download.prompt_for_download': False,
    'download.directory_upgrade': True,
    'safebrowsing.enabled': True,
    'profile.default_content_setting_values.automatic_downloads': 1,
})
chrome_options.add_experimental_option('excludeSwitches', ['enable-logging', 'enable-automation', 'enable-sync'])
chrome_options.add_experimental_option('useAutomationExtension', False)
for arg in [
    '--disable-blink-features=AutomationControlled','--disable-notifications','--disable-infobars',
    '--disable-popup-blocking','--disable-save-password-bubble',
    '--disable-features=IsolateOrigins,site-per-process,TranslateUI','--disable-site-isolation-trials',
    '--no-first-run','--no-service-autorun','--disable-default-apps','--disable-sync',
    '--disable-translate','--disable-extensions','--disable-background-timer-throttling',
    '--disable-client-side-phishing-detection','--disable-component-extensions-with-background-pages',
    '--disable-hang-monitor','--disable-prompt-on-repost','--metrics-recording-only',
]:
    chrome_options.add_argument(arg)

chrome_options.page_load_strategy = 'normal'

driver = webdriver.Chrome(options=chrome_options)
driver.set_page_load_timeout(30)
driver.set_script_timeout(30)
driver.maximize_window()
wait = WebDriverWait(driver, 30)


In [ ]:
# Acessar site e realizar login
def iniciar_login(URL_LOGIN: str, EMAIL: str, SENHA: str, timeout: int = 30):
    if driver is None:
        raise RuntimeError("O driver global não foi inicializado. Execute a Célula 1 primeiro.")
    _wait = WebDriverWait(driver, timeout)

    print("✓ Iniciando processo de login...")
    driver.get(URL_LOGIN)
    time.sleep(2)

    campo_email = _wait.until(EC.presence_of_element_located((By.ID, "login")))
    campo_email.clear()
    campo_email.send_keys(EMAIL)
    print("✓ Email inserido!")

    campo_senha = driver.find_element(By.NAME, "senha")
    campo_senha.clear()
    campo_senha.send_keys(SENHA)
    print("✓ Senha inserida!")

    botao_login = driver.find_element(By.XPATH, "//button[@type='submit']")
    botao_login.click()
    print("✓ Login clicado!")
    time.sleep(3)

    def buscar_codigo_outlook():
        print("Aguardando 30 s para recebimento do código...")
        time.sleep(30)
        outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")
        inbox = outlook.GetDefaultFolder(6)
        messages = inbox.Items
        messages.Sort("[ReceivedTime]", True)
        for message in list(messages)[:10]:
            try:
                if "CÓDIGO DE VALIDAÇÃO" in str(message.Subject).upper():
                    body = str(message.Body)
                    codigo_match = re.search(r'Código de ativação:\s*(\d{6})', body, re.IGNORECASE)
                    if codigo_match:
                        return codigo_match.group(1)
            except Exception:
                continue
        return None

    codigo = buscar_codigo_outlook()
    if not codigo:
        raise ValueError("Não foi possível encontrar o código de validação no Outlook.")
    print(f"✓ Código obtido: {codigo}")

    campo_codigo = _wait.until(EC.presence_of_element_located((By.ID, "code")))
    campo_codigo.clear()
    campo_codigo.send_keys(codigo)
    print("✓ Código inserido!")

    botao_salvar = _wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@onclick='salvar()']")))
    botao_salvar.click()
    print("✓ Logado com sucesso!")
    time.sleep(3)


In [ ]:
# ── Liberação de Cursos ────
from selenium.webdriver.support.ui import Select

URL_LIBERA_CURSOS = "https://gehealthcare.sistematutor.com.br/tutor/sistema/matricula.do?method=pesquisar"
ARQUIVO_ACESSOS = "AcessosEspeciais.xlsx"


def normalizar_texto(valor) -> str:
    if pd.isna(valor):
        return ""
    texto = str(valor).strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    texto = re.sub(r"\s+", " ", texto)
    return texto.upper()


def numeros_unicos(valores):
    saida = []
    for valor in valores:
        valor = str(valor).strip()
        if valor and valor not in saida:
            saida.append(valor)
    return saida


def detectar_idioma(produto_1: str) -> str:
    texto = normalizar_texto(produto_1)
    if "ESP" in texto or "IMSS" in texto or "PRODUTOS PADRAO (ESP)" in texto:
        return "ESP"
    if "PT" in texto or "FLEURY" in texto or "PRODUTOS PADRAO (PT)" in texto:
        return "PT"
    return "PT"


MAPA_MODALIDADE_SITE = {
    "PT": {
        "RESSONANCIA": ["366", "288"],
        "RESONANCIA": ["366", "288"],
        "TOMOGRAFIA": ["359", "565"],
        "CIRURGIA": ["312"],
        "CIRUGIA": ["312"],
        "INTERVENCAO": ["567", "613"],
        "INTERVENCION": ["567", "613"],
        "MAMOGRAFIA": ["499", "507"],
        "MAMMOGRAFIA": ["499", "507"],
        "MAMOGRAFIA / MAMOGRAFIA": ["499", "507"],
        "SAUDE OSSEA": ["283", "289"],
        "SALUD OSSEA": ["283", "289"],
        "MEDICINA NUCLEAR": ["602", "125", "94", "61"],
        "RAIOS X": ["178", "179", "180", "181", "93", "155"],
        "RAYOS X": ["178", "179", "180", "181", "93", "155"],
        "RAIO X": ["178", "179", "180", "181", "93", "155"],
    },
    "ESP": {
        "RESSONANCIA": ["279", "339"],
        "RESONANCIA": ["279", "339"],
        "TOMOGRAFIA": ["282", "207", "579"],
        "CIRURGIA": ["308"],
        "CIRUGIA": ["308"],
        "INTERVENCAO": ["121", "158", "159", "559"],
        "INTERVENCION": ["121", "158", "159", "559"],
        "MAMOGRAFIA": ["500"],
        "MAMMOGRAFIA": ["500"],
        "MAMOGRAFIA / MAMOGRAFIA": ["500"],
        "SAUDE OSSEA": ["327", "290"],
        "SALUD OSSEA": ["327", "290"],
        "MEDICINA NUCLEAR": ["602", "125", "94", "61"],
        "RAIOS X": ["178", "179", "180", "181", "93", "155"],
        "RAYOS X": ["178", "179", "180", "181", "93", "155"],
        "RAIO X": ["178", "179", "180", "181", "93", "155"],
    },
}


def dividir_modalidades(modalidade) -> list:
    if pd.isna(modalidade):
        return []
    texto = str(modalidade).strip()
    if not texto:
        return []
    partes_linha = [p.strip() for p in re.split(r"[\n;|]+", texto) if p.strip()]
    if not partes_linha:
        partes_linha = [texto]
    return partes_linha


def cursos_por_modalidade(parte_modalidade: str, idioma: str) -> list:
    parte_norm = normalizar_texto(parte_modalidade)
    if "INTERVEN" in parte_norm and "IGS" in parte_norm:
        cursos = []
        if "7" in parte_norm:
            cursos.extend(["613"])
        if "3" in parte_norm or "5" in parte_norm:
            cursos.extend(["567", "568"])
        return numeros_unicos(cursos)

    mapa_site = MAPA_MODALIDADE_SITE.get(idioma, MAPA_MODALIDADE_SITE["PT"])
    for chave, cursos in mapa_site.items():
        chave_norm = normalizar_texto(chave)
        if chave_norm in parte_norm or parte_norm in chave_norm:
            return list(cursos)
    return []


def extrair_cursos(produto_1, modalidade="", instituicao="") -> list:
    texto_produto = str(produto_1) if pd.notna(produto_1) else ""
    texto_instituicao = str(instituicao) if pd.notna(instituicao) else ""
    combinado_norm = normalizar_texto(f"{texto_produto} {texto_instituicao}")

    if "IMSS" in combinado_norm or "IMMS" in combinado_norm or "INSTITUTO MEXICANO DEL SEGURO SOCIAL" in combinado_norm:
        return ["652", "656", "658"]
    if "FLEURY" in combinado_norm:
        return ["475"]

    numeros_produto = re.findall(r"\d+", texto_produto)
    if numeros_produto:
        return numeros_unicos(numeros_produto)

    idioma = detectar_idioma(texto_produto)
    partes = dividir_modalidades(modalidade)

    cursos = []
    for parte in partes[:2]:
        nums = re.findall(r"\d+", parte)
        if nums:
            cursos.extend(nums)
            continue
        cursos_encontrados = cursos_por_modalidade(parte, idioma)
        if cursos_encontrados:
            cursos.extend(cursos_encontrados)
        else:
            print(f"⚠ Modalidade sem mapeamento automático ({idioma}): {parte}")
    return numeros_unicos(cursos)


def carregar_base_liberacao(nome_arquivo: str = ARQUIVO_ACESSOS_ESPECIAIS) -> pd.DataFrame:
    """
    Versão para a planilha AcessosEspeciais.xlsx.
    Colunas esperadas: NOME COMPLETO | EMAIL | INSTITUIÇÃO | GON | PRODUTO

    - Aplica idioma fixo IDIOMA_FIXO (PT por padrão).
    - Cursos a liberar = CURSOS_FIXOS definido na célula de configuração
      (ignora coluna PRODUTO da planilha; pode-se trocar para usar a coluna
      se desejar futuramente).
    """
    caminho = os.path.join(os.getcwd(), nome_arquivo)
    df = pd.read_excel(caminho)
    df.columns = df.columns.str.strip()

    # Normalizar colunas para o padrão usado pelo bot
    rename_map = {
        "NOME COMPLETO": "Nome",
        "EMAIL": "E-mail",
        "INSTITUIÇÃO": "Instituição",
        "GON": "GON",
        "PRODUTO": "Produto",
    }
    df = df.rename(columns=rename_map)

    # Filtra e-mails válidos
    df = df[df["E-mail"].notna()].copy()
    df["E-mail"] = df["E-mail"].astype(str).str.strip()
    df = df[df["E-mail"] != ""].copy()
    df = df.drop_duplicates(subset=["E-mail"], keep="first").reset_index(drop=True)

    # Idioma fixo
    df["Idioma"] = IDIOMA_FIXO

    # Cursos fixos para todos os alunos
    cursos = [str(c).strip() for c in CURSOS_FIXOS if str(c).strip()]
    df["Cursos Solicitados"] = [list(cursos) for _ in range(len(df))]
    df["Cursos Solicitados Texto"] = ", ".join(cursos)

    # Campos auxiliares (mantidos para compatibilidade com selecionar_cursos)
    if "Produto 1" not in df.columns:
        df["Produto 1"] = ""
    if "Modalidade" not in df.columns:
        df["Modalidade"] = ""

    df["Status Liberação"] = ""
    df["Detalhe Liberação"] = ""

    print(f"✓ Base carregada para liberação: {len(df)} aluno(s). "
          f"Cursos fixos: {', '.join(cursos)} | Site: {IDIOMA_FIXO}")
    return df


def aceitar_alerta_se_existir(driver, timeout=2):
    try:
        alerta = WebDriverWait(driver, timeout).until(EC.alert_is_present())
        texto = alerta.text
        alerta.accept()
        print(f"⚠ Alerta aceito: {texto}")
        return texto
    except TimeoutException:
        return None
    except NoAlertPresentException:
        return None


def pesquisar_matriculas(email: str, _wait):
    driver.get(URL_LIBERA_CURSOS)
    time.sleep(2)
    campo = _wait.until(EC.presence_of_element_located((By.ID, "nome1")))
    campo.clear()
    campo.send_keys(email)
    campo.send_keys(Keys.ENTER)
    time.sleep(3)


def cursos_ja_liberados_na_tabela(_wait) -> set:
    tabela = _wait.until(EC.presence_of_element_located((By.ID, "tabelaAlunos")))
    linhas = tabela.find_elements(By.CSS_SELECTOR, "tbody tr")
    cursos = set()
    for linha in linhas:
        if "Nenhum registro encontrado" in linha.text:
            continue
        colunas = linha.find_elements(By.TAG_NAME, "td")
        if len(colunas) >= 4:
            texto_curso = colunas[3].text.strip()
            cursos.update(re.findall(r"\d+", texto_curso))
    return cursos


def abrir_nova_matricula(_wait):
    botao = _wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//input[@value='Incluir uma Nova Matrícula em Produtos']")
    ))
    botao.click()
    time.sleep(2)


def selecionar_aluno_por_email(email: str, _wait):
    campo = _wait.until(EC.presence_of_element_located((By.ID, "nome1")))
    campo.clear()
    campo.send_keys(email)

    resultado = _wait.until(EC.visibility_of_element_located((By.ID, "resultadoAlunos")))
    primeiro = resultado.find_element(By.CSS_SELECTOR, "#listaDinamica li")
    primeiro.click()
    print("  ✓ Aluno selecionado no autocomplete")
    time.sleep(1)


def selecionar_site(idioma: str, _wait):
    valor_site = "2" if idioma == "ESP" else "1"
    select_empresa = _wait.until(EC.presence_of_element_located((By.ID, "idEmpresa")))
    Select(select_empresa).select_by_value(valor_site)
    print(f"  ✓ Site selecionado: {'Espanhol' if idioma == 'ESP' else 'Português'}")
    time.sleep(2)


def palavras_contexto(produto_1="", modalidade="", idioma="") -> list:
    texto = normalizar_texto(f"{produto_1} {modalidade}")
    chaves = []
    regras = [
        ("MAMOGRAF", ["MAMOGRAF", "MAMMOGRAF"]),
        ("RESSON", ["RESSON", "RESON"]),
        ("TOMOGRAF", ["TOMOGRAF", "CT"]),
        ("SAUDE OSSEA", ["OSSEA", "DENSITOMETRIA"]),
        ("SALUD OSSEA", ["OSSEA", "SALUD"]),
        ("MEDICINA NUCLEAR", ["NUCLEAR"]),
        ("RAIO", ["RAIO", "RAYOS", "X RAY", "RX"]),
        ("RAYOS", ["RAIO", "RAYOS", "X RAY", "RX"]),
        ("SURGERY", ["SURGERY", "CIRURG", "CIRUG"]),
        ("CIRURG", ["SURGERY", "CIRURG", "CIRUG"]),
        ("INTERVEN", ["INTERVEN", "IGS"]),
        ("IMSS", ["IMSS"]),
        ("FLEURY", ["FLEURY"]),
    ]
    for gatilho, termos in regras:
        if gatilho in texto:
            chaves.extend(termos)
    if idioma == "ESP":
        chaves.extend(["ESP", "ESPANHOL", "ESPANOL", "SPANISH"])
    else:
        chaves.extend(["PT", "PORTUGUES", "PORTUGUESE"])
    return numeros_unicos(chaves)


def selecionar_cursos(cursos: list, _wait, produto_1="", modalidade="", idioma=""):
    """Seleciona 1 opção por código no select de produtos, com priorização por contexto."""
    select_elem = _wait.until(EC.presence_of_element_located((By.ID, "idCurso")))
    contexto = palavras_contexto(produto_1, modalidade, idioma)

    selecionados = []
    detalhes = []

    driver.execute_script(
        """
        const select = arguments[0];
        for (const opt of select.options) { opt.selected = false; }
        select.dispatchEvent(new Event('change', {bubbles: true}));
        """,
        select_elem,
    )

    for curso in cursos:
        resultado = driver.execute_script(
            r"""
            const select = arguments[0];
            const curso = String(arguments[1]).trim();
            const contexto = arguments[2] || [];

            function norm(s) {
                return String(s || '').normalize('NFD').replace(/[\u0300-\u036f]/g, '').toUpperCase();
            }
            function temCodigoIsolado(texto, codigo) {
                const re = new RegExp('(^|[^0-9])' + codigo.replace(/[.*+?^${}()|[\]\\]/g, '\\$&') + '([^0-9]|$)');
                return re.test(texto);
            }

            const candidatos = [];
            for (let i = 0; i < select.options.length; i++) {
                const opt = select.options[i];
                const value = String(opt.value || '').trim();
                const text = String(opt.text || '').trim();
                const textNorm = norm(text);
                let score = -1;

                if (value === curso) {
                    score = 1000;
                } else if (temCodigoIsolado(text, curso)) {
                    score = 700;
                } else {
                    continue;
                }

                for (const termo of contexto) {
                    const termoNorm = norm(termo);
                    if (termoNorm && textNorm.includes(termoNorm)) {
                        score += 30;
                    }
                }
                if (textNorm.includes('TESTE') || textNorm.includes('MIGRACAO') || textNorm.includes('ARQUIV')) {
                    score -= 200;
                }
                candidatos.push({index: i, value, text, score});
            }

            candidatos.sort((a, b) => b.score - a.score);
            if (candidatos.length === 0) {
                return {found: false, curso, candidatos: []};
            }
            const escolhido = candidatos[0];
            select.options[escolhido.index].selected = true;
            select.dispatchEvent(new Event('change', {bubbles: true}));
            return {found: true, curso, escolhido, candidatos: candidatos.slice(0, 5)};
            """,
            select_elem,
            curso,
            contexto,
        )

        if resultado and resultado.get("found"):
            selecionados.append(curso)
            escolhido = resultado.get("escolhido", {})
            detalhes.append(f"{curso} -> {escolhido.get('text', '')}")
            print(f"  ✓ Curso selecionado: {curso} | {escolhido.get('text', '')}")
        else:
            print(f"  ⚠ Curso não encontrado no dropdown: {curso}")

    if not selecionados:
        raise RuntimeError("Nenhum dos cursos solicitados foi encontrado no dropdown de cursos.")

    try:
        driver.execute_script("$('#idCurso').selectpicker('refresh');")
    except Exception:
        pass

    botao_adicionar = _wait.until(EC.presence_of_element_located((By.ID, "btnAdicionar")))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", botao_adicionar)
    time.sleep(0.5)
    try:
        botao_adicionar.click()
    except ElementClickInterceptedException:
        driver.execute_script("arguments[0].click();", botao_adicionar)

    print("  ✓ Cursos adicionados")
    time.sleep(1)
    return selecionados, detalhes



def remover_cursos_indesejados(cursos_esperados: list, _wait, timeout: int = 8) -> list:
    """
    Após o Adicionar, lê o select itemProduto1 (lista de cursos já adicionados),
    identifica quaisquer cursos cujo código numérico NÃO esteja na lista esperada,
    seleciona apenas esses e clica no btnRemover para devolvê-los ao pool.
    Retorna a lista de cursos removidos (códigos).
    """
    esperados_set = {str(c).strip() for c in cursos_esperados}
    try:
        item_select = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.ID, "itemProduto1"))
        )
    except TimeoutException:
        print("  ⚠ itemProduto1 não encontrado para reconciliação.")
        return []

    # Extrai códigos atuais e desmarca tudo
    info = driver.execute_script(
        r"""
        const select = arguments[0];
        const esperados = new Set(arguments[1].map(String));
        const atuais = [];
        const removerIdx = [];
        for (let i = 0; i < select.options.length; i++) {
            const opt = select.options[i];
            opt.selected = false;
            const value = String(opt.value || '').trim();
            const text  = String(opt.text  || '').trim();
            // tenta value direto; se não bater, busca números no texto
            let codigos = [];
            if (value) codigos.push(value);
            const matchs = text.match(/\d+/g) || [];
            for (const m of matchs) codigos.push(m);
            // dedup
            codigos = [...new Set(codigos)];
            atuais.push({index: i, value, text, codigos});
            // se NENHUM código bate com a lista esperada -> remover
            const algumEsperado = codigos.some(c => esperados.has(c));
            if (!algumEsperado) {
                opt.selected = true;
                removerIdx.push({index: i, value, text});
            }
        }
        select.dispatchEvent(new Event('change', {bubbles: true}));
        return {atuais, removerIdx};
        """,
        item_select,
        list(esperados_set),
    )

    atuais = info.get("atuais", [])
    a_remover = info.get("removerIdx", [])
    print(f"  • itemProduto1 contém {len(atuais)} curso(s); marcados para remoção: {len(a_remover)}")
    for r in a_remover:
        print(f"    - remover: {r.get('value')} | {r.get('text')}")

    if not a_remover:
        return []

    try:
        botao_remover = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.ID, "btnRemover"))
        )
    except TimeoutException:
        print("  ⚠ btnRemover não encontrado.")
        return []

    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", botao_remover)
    time.sleep(0.4)
    try:
        botao_remover.click()
    except ElementClickInterceptedException:
        driver.execute_script("arguments[0].click();", botao_remover)
    time.sleep(0.8)

    try:
        driver.execute_script("$('#idCurso').selectpicker('refresh');")
    except Exception:
        pass

    removidos = [r.get("value") or "" for r in a_remover]
    print(f"  ✓ {len(removidos)} curso(s) indesejado(s) removido(s)")
    return removidos


def preencher_condicoes_comerciais(_wait):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)

    qtd = _wait.until(EC.presence_of_element_located((By.ID, "quantidadeProduto")))
    qtd.clear()
    qtd.send_keys("1")
    qtd.send_keys(Keys.TAB)
    print("  ✓ Quantidade preenchida: 1")

    forma = _wait.until(EC.presence_of_element_located((By.ID, "formaPagamentoSelecionado")))
    Select(forma).select_by_value("U-19")
    print("  ✓ Forma de pagamento: Gratuito")
    time.sleep(1)

    parcela = _wait.until(EC.presence_of_element_located((By.NAME, "parcelaSelecionada")))
    Select(parcela).select_by_value("1")
    print("  ✓ Parcela selecionada: 1")

    responsavel = _wait.until(EC.presence_of_element_located((By.NAME, "auxResponsavel")))
    Select(responsavel).select_by_value("1")
    print("  ✓ Responsável financeiro: O Próprio Aluno")
    time.sleep(1)

    # ── Preenchimento robusto do campo Desconto ─────────────────────────────
    # O campo tem dois handlers críticos:
    #   onfocus -> copia valor atual para tempValorFormPercentualDesconto1
    #   onblur  -> SE temp != novo valor, chama selecionaDesconto(1)
    # Se simplesmente fizermos set value via JS, o onfocus nunca rodou e o
    # onblur compara temp ("") com "100", o que normalmente até funciona,
    # mas em alguns fluxos o campo vem com 0/100 pré-preenchido e perdemos
    # o disparo. Por isso forçamos: scroll -> focus (dispara onfocus) ->
    # selecionar todo conteúdo -> apagar -> digitar 100 -> blur
    # -> chamar selecionaDesconto(1) explicitamente como fallback.

    desconto = _wait.until(EC.presence_of_element_located((By.ID, "valorFormPercentualDesconto1")))

    # 1) garante visibilidade no viewport
    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'}); window.scrollBy(0, -60);",
        desconto,
    )
    time.sleep(0.4)

    # 2) foco explícito pelo Selenium para disparar onfocus (que popula tempValorFormPercentualDesconto1)
    try:
        ActionChains(driver).move_to_element(desconto).click().perform()
    except Exception:
        driver.execute_script("arguments[0].focus();", desconto)
    time.sleep(0.3)

    # 3) seleciona tudo + apaga conteúdo existente + digita 100
    desconto.send_keys(Keys.CONTROL, "a")
    desconto.send_keys(Keys.DELETE)
    time.sleep(0.2)
    desconto.send_keys("100")
    time.sleep(0.3)

    # 4) tira o foco para disparar o onblur (que chamará selecionaDesconto(1))
    desconto.send_keys(Keys.TAB)
    time.sleep(0.8)

    # 5) Validação: lê o value via JS. Se não estiver "100", força via JS e dispara eventos.
    valor_atual = driver.execute_script(
        "return document.getElementById('valorFormPercentualDesconto1').value;"
    )
    if str(valor_atual).strip() not in ("100", "100.00", "100,00"):
        print(f"  ⚠ Desconto não ficou em 100 (valor atual: {valor_atual!r}). Forçando via JS...")
        driver.execute_script(
            """
            const el = document.getElementById('valorFormPercentualDesconto1');
            const form = document.forms['MatriculaForm'];
            // simula onfocus: salva valor anterior
            if (form && form.tempValorFormPercentualDesconto1) {
                form.tempValorFormPercentualDesconto1.value = el.value || '';
            }
            el.focus();
            el.value = '100';
            el.dispatchEvent(new Event('input',  {bubbles: true}));
            el.dispatchEvent(new Event('change', {bubbles: true}));
            // marca foco percentual como esperado pelo onblur do site
            if (form && form.focoPercentual1) { form.focoPercentual1.value = 'P'; }
            el.blur();
            el.dispatchEvent(new Event('blur', {bubbles: true}));
            // fallback final: chama a função do site diretamente
            if (typeof selecionaDesconto === 'function') {
                try { selecionaDesconto(1); } catch (e) { console.log('selecionaDesconto erro:', e); }
            }
            """
        )
        time.sleep(0.8)
        valor_atual = driver.execute_script(
            "return document.getElementById('valorFormPercentualDesconto1').value;"
        )

    # 6) move o foco para outro campo neutro para garantir blur final
    try:
        driver.execute_script(
            "document.getElementById('quantidadeProduto').focus(); "
            "document.getElementById('quantidadeProduto').blur();"
        )
    except Exception:
        pass

    print(f"  ✓ Desconto preenchido: {valor_atual}%")
    time.sleep(1.0)


def salvar_liberacao(_wait, timeout_total: int = 25):
    """
    Clica no botão Salvar de forma robusta, lidando com:
      - overlays/dropdowns do bootstrap-select interceptando o clique
      - foco em campos com onblur ainda processando
      - botão fora da viewport
      - múltiplos alertas de confirmação após o salvar
      - redirecionamento (mudança de URL) após o submit
    """
    url_antes = driver.current_url

    # 1) Garante que nenhum dropdown bootstrap-select esteja aberto sobre a tela
    try:
        driver.execute_script("""
            // Fecha qualquer bootstrap-select aberto
            try { $('.bootstrap-select.open').removeClass('open'); } catch(e){}
            try { $('.bootstrap-select.show').removeClass('show'); } catch(e){}
            try { $('.dropdown-menu.show').removeClass('show'); } catch(e){}
            // Tira o foco de qualquer campo com onblur pendente
            if (document.activeElement && document.activeElement.blur) {
                document.activeElement.blur();
            }
        """)
    except Exception:
        pass
    time.sleep(0.8)

    # 2) Localiza o botão e rola para ele com offset (evita header/footer fixos)
    botao_salvar = _wait.until(EC.presence_of_element_located((By.ID, "btnSalvar")))
    driver.execute_script("""
        const el = arguments[0];
        el.scrollIntoView({block: 'center', inline: 'center'});
        window.scrollBy(0, -80);
    """, botao_salvar)
    time.sleep(0.8)

    # 3) Tenta a sequência de cliques: nativo -> JS click -> ActionChains -> chamar salvar()
    clicado = False
    erros = []

    # Tentativa A: clique nativo aguardando clickable
    try:
        WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.ID, "btnSalvar"))).click()
        clicado = True
        print("  ✓ Salvar clicado (nativo)")
    except Exception as e:
        erros.append(f"nativo: {type(e).__name__}")

    # Tentativa B: JS click no elemento
    if not clicado:
        try:
            driver.execute_script("arguments[0].click();", botao_salvar)
            clicado = True
            print("  ✓ Salvar clicado (JS click)")
        except Exception as e:
            erros.append(f"jsclick: {type(e).__name__}")

    # Tentativa C: ActionChains move + click
    if not clicado:
        try:
            ActionChains(driver).move_to_element(botao_salvar).pause(0.3).click().perform()
            clicado = True
            print("  ✓ Salvar clicado (ActionChains)")
        except Exception as e:
            erros.append(f"actions: {type(e).__name__}")

    # Tentativa D: chamar a função salvar() diretamente
    if not clicado:
        try:
            driver.execute_script("if (typeof salvar === 'function') { salvar(); }")
            clicado = True
            print("  ✓ Salvar disparado (chamada direta a salvar())")
        except Exception as e:
            erros.append(f"salvar(): {type(e).__name__}")

    if not clicado:
        raise RuntimeError(f"Não foi possível clicar em Salvar. Tentativas: {erros}")

    # 4) Aceita alertas de confirmação que aparecem após o submit (ex.: "Confirma?", "Salvo com sucesso").
    alertas = []
    deadline = time.time() + timeout_total
    while time.time() < deadline:
        texto = aceitar_alerta_se_existir(driver, timeout=2)
        if texto:
            alertas.append(texto)
            time.sleep(0.8)
            continue
        # Sem alerta no momento; se a URL mudou ou voltou pra tela de pesquisa, sucesso
        try:
            url_atual = driver.current_url
        except Exception:
            url_atual = url_antes
        if url_atual != url_antes:
            print(f"  ✓ Salvar concluído (URL mudou para {url_atual})")
            break
        # Verifica se estamos de volta na pesquisa de matrículas
        try:
            if driver.find_elements(By.ID, "tabelaAlunos"):
                print("  ✓ Salvar concluído (retornou à tela de pesquisa)")
                break
        except Exception:
            pass
        time.sleep(1)

    # 5) Última varredura por alertas residuais
    for _ in range(3):
        texto = aceitar_alerta_se_existir(driver, timeout=1)
        if texto:
            alertas.append(texto)
            time.sleep(0.5)
        else:
            break

    return " | ".join(alertas)


def liberar_cursos(df: pd.DataFrame, timeout: int = 30) -> pd.DataFrame:
    _wait = WebDriverWait(driver, timeout)
    total = len(df)

    for contador, (idx, row) in enumerate(df.iterrows(), start=1):
        email = str(row["E-mail"]).strip()
        idioma = row["Idioma"]
        cursos_solicitados = list(row["Cursos Solicitados"])

        print(f"\n{'='*70}")
        print(f"[{contador}/{total}] Liberando cursos para: {email} | Site: {idioma} | Cursos: {', '.join(cursos_solicitados)}")

        try:
            aceitar_alerta_se_existir(driver, timeout=1)
            pesquisar_matriculas(email, _wait)
            cursos_existentes = cursos_ja_liberados_na_tabela(_wait)
            faltantes = [curso for curso in cursos_solicitados if curso not in cursos_existentes]

            if not faltantes:
                print(f"✓ Usuário já possui todos os cursos solicitados: {email}")
                df.at[idx, "Status Liberação"] = "Cursos já liberados"
                df.at[idx, "Detalhe Liberação"] = f"Já existentes: {', '.join(cursos_solicitados)}"
                continue

            print(f"→ Cursos faltantes: {', '.join(faltantes)}")
            abrir_nova_matricula(_wait)
            selecionar_aluno_por_email(email, _wait)
            selecionar_site(idioma, _wait)
            cursos_selecionados, detalhes_cursos = selecionar_cursos(
                faltantes, _wait,
                produto_1=row.get("Produto 1", ""),
                modalidade=row.get("Modalidade", ""),
                idioma=idioma,
            )
            removidos = remover_cursos_indesejados(faltantes, _wait)
            preencher_condicoes_comerciais(_wait)
            alertas = salvar_liberacao(_wait)

            df.at[idx, "Status Liberação"] = "Cursos liberados agora"
            df.at[idx, "Detalhe Liberação"] = f"Liberados: {', '.join(cursos_selecionados)} | Seleções: {'; '.join(detalhes_cursos)}" + (f" | Removidos: {', '.join(removidos)}" if removidos else "")
            if alertas:
                df.at[idx, "Detalhe Liberação"] += f" | Alertas: {alertas}"

        except UnexpectedAlertPresentException:
            alerta = aceitar_alerta_se_existir(driver, timeout=3)
            print(f"⚠ Alerta inesperado aceito: {alerta}")
            df.at[idx, "Status Liberação"] = "Erro - Alerta inesperado"
            df.at[idx, "Detalhe Liberação"] = str(alerta)
            continue

        except Exception as e:
            print(f"✗ Erro ao liberar cursos para {email}: {e}")
            df.at[idx, "Status Liberação"] = "Erro - Liberação"
            df.at[idx, "Detalhe Liberação"] = str(e)
            continue

    return df


In [ ]:
print("\n=== [1] Iniciando login no sistema ===")
iniciar_login(URL_LOGIN, EMAIL, SENHA)

print("\n=== [2] Carregando base de acessos especiais ===")
df = carregar_base_liberacao(ARQUIVO_ACESSOS_ESPECIAIS)

print("\n=== [3] Liberando cursos ===")
df_resultado = liberar_cursos(df)

df_resultado.to_excel("Resultado_LiberacaoEspecial.xlsx", index=False)
print("\n✓ Resultado salvo em Resultado_LiberacaoEspecial.xlsx")
